In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!cp /content/drive/MyDrive/Data_Science_Project/data/musan.tar.gz /content/
!tar -xf musan.tar.gz
!rm musan.tar.gz

In [ ]:
!pip install torchaudio speechbrain webrtc_noise_gain

In [ ]:
import glob, random, zipfile, io, time, shutil, os, threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import soundfile as sf
import webrtc_noise_gain as wng

SAMPLE_RATE = 16000
EMBEDDING_DIM = 192
MAX_SAMPLES = 160000  # 10 seconds at 16kHz

FEATURE_ORDER = [
  "log_duration", "voicing_probability", "spectral_centroid", "cpp",
  "wada_snr", "zero_crossing_rate", "spectral_flatness", "subband_energy_ratio",
]

_WEBRTC_PROCESSOR = None

_OPEN_ZIPS = {}
_DRIVE_FS = None
_ZIP_LOCK = threading.Lock() # CRITICAL: ZipFile reading is not thread-safe

_WADA_ALPHA = 0.4
_WADA_TABLE_V = None
_WADA_TABLE_SNR = None

In [ ]:
def _init_webrtc():
  global _WEBRTC_PROCESSOR
  if _WEBRTC_PROCESSOR is None:
    _WEBRTC_PROCESSOR = wng.AudioProcessor(noise_suppression=4)
  return _WEBRTC_PROCESSOR

def enhance_waveform_cpu(waveform: torch.Tensor, return_vad: bool = False):
  processor = _init_webrtc()
  int16_data = (waveform * 32767).clamp(-32768, 32767).short().numpy().tobytes()
  frame_size_bytes = 320
  frames = [int16_data[i:i+frame_size_bytes] for i in range(0, len(int16_data), frame_size_bytes)]
  out_bytes = bytearray()
  speech_flags = []

  for frame in frames:
    if len(frame) < frame_size_bytes:
      frame += b'\x00' * (frame_size_bytes - len(frame))
    res = processor.Process10ms(frame)
    out_bytes += res.audio if res.is_speech else frame
    speech_flags.append(bool(res.is_speech))

  out_np = np.frombuffer(out_bytes, dtype=np.int16).astype(np.float32) / 32767.0
  out_tensor = torch.from_numpy(out_np)

  if out_tensor.shape[-1] < waveform.shape[-1]:
    out_tensor = F.pad(out_tensor, (0, waveform.shape[-1] - out_tensor.shape[-1]))
  else:
    out_tensor = out_tensor[:waveform.shape[-1]]

  if out_tensor.shape[-1] > MAX_SAMPLES:
    out_tensor = out_tensor[:MAX_SAMPLES]

  if return_vad:
    return out_tensor, np.array(speech_flags, dtype=bool)

  return out_tensor

In [ ]:
def configure_drive_filesystem(fs):
  global _DRIVE_FS
  _DRIVE_FS = fs

def _get_zip_handle(zip_path: str):
  global _DRIVE_FS

  if zip_path not in _OPEN_ZIPS:
    if _DRIVE_FS is None and 'fs' in globals():
      _DRIVE_FS = globals()['fs']
    if _DRIVE_FS is not None:
      file_obj = _DRIVE_FS.open(zip_path, "rb")
      _OPEN_ZIPS[zip_path] = zipfile.ZipFile(file_obj, "r")
    else:
      _OPEN_ZIPS[zip_path] = zipfile.ZipFile(zip_path, "r")

  return _OPEN_ZIPS[zip_path]

In [ ]:
def load_mono_16k(path: str) -> torch.Tensor:
  waveform, sr = torchaudio.load(path)

  if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)
  if sr != SAMPLE_RATE:
    waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)
  if waveform.shape[-1] > MAX_SAMPLES:
    waveform = waveform[:, :MAX_SAMPLES]

  return waveform.squeeze(0)

def load_mono_16k_from_zip(file_ref, target_sr=SAMPLE_RATE):
  zip_path, member_name = file_ref

  # Thread-safe read from the central ZipFile object
  with _ZIP_LOCK:
      zf = _get_zip_handle(zip_path)
      raw_bytes = zf.read(member_name)

  buf = io.BytesIO(raw_bytes)
  try:
      waveform, sr = torchaudio.load(buf)
  except Exception:
      buf.seek(0)
      data, sr = sf.read(buf, dtype='float32')
      waveform = torch.from_numpy(data).unsqueeze(0) if data.ndim == 1 else torch.from_numpy(data.T)
  if waveform.shape[0] > 1:
      waveform = waveform.mean(dim=0, keepdim=True)
  if sr != target_sr:
      waveform = torchaudio.functional.resample(waveform, sr, target_sr)
  if waveform.shape[-1] > MAX_SAMPLES:
      waveform = waveform[:, :MAX_SAMPLES]
  return waveform.squeeze(0)

In [ ]:
def _rms(x):
    return torch.sqrt(torch.mean(x ** 2) + 1e-12)

def add_noise_at_snr(clean, noise_path, snr_db, rng):
    noise = load_mono_16k(noise_path)
    if noise.shape[-1] < clean.shape[-1]:
        reps = int(np.ceil(clean.shape[-1] / noise.shape[-1]))
        noise = noise.repeat(reps)

    if noise.shape[-1] > clean.shape[-1]:
        max_start = noise.shape[-1] - clean.shape[-1]
        start = rng.randint(0, max_start)
        noise = noise[start:start + clean.shape[-1]]

    target_noise_rms = _rms(clean) / (10 ** (snr_db / 20))
    noise = noise * (target_noise_rms / (_rms(noise) + 1e-8))
    mixed = clean + noise

    if mixed.shape[-1] > MAX_SAMPLES:
        mixed = mixed[:MAX_SAMPLES]

    return mixed

In [ ]:
def per_file_rng(condition: str, file_path: str, base_seed: int) -> random.Random:
    import zlib
    cond_offset = zlib.crc32(condition.encode("utf-8")) ^ base_seed
    return random.Random(zlib.crc32(f"{condition}:{file_path}".encode()) ^ cond_offset)

def get_embedding(signal_tensor, model, device):
    signal = signal_tensor.unsqueeze(0).float().to(device)

    with torch.no_grad():
        emb = model.encode_batch(signal).squeeze(1)
        return F.normalize(emb, p=2, dim=-1).squeeze(0).cpu()

In [ ]:
def _build_wada_snr_table(alpha=0.4, snr_range=(-20, 40), step=1.0, mc_samples=200_000, seed=0):
    rng = np.random.RandomState(seed)
    snrs = np.arange(snr_range[0], snr_range[1] + step, step)
    theta = 1.0
    clean_second_moment = theta ** 2 * alpha * (alpha + 1)
    g = rng.gamma(alpha, theta, size=mc_samples)
    sign = rng.choice([-1.0, 1.0], size=mc_samples)
    s = g * sign
    v_vals = np.empty_like(snrs, dtype=np.float64)
    for i, snr_db in enumerate(snrs):
        sigma = np.sqrt(clean_second_moment) * 10 ** (-snr_db / 20.0)
        n = rng.normal(0.0, sigma, size=mc_samples)
        y = s + n
        v_vals[i] = (np.mean(np.abs(y)) ** 2) / (np.mean(y ** 2) + 1e-12)
    order = np.argsort(v_vals)
    return v_vals[order], snrs[order]

_WADA_TABLE_V, _WADA_TABLE_SNR = _build_wada_snr_table()

In [ ]:
def compute_spectral_quality_features_batch(waveforms: torch.Tensor, lengths: torch.Tensor, vads_mean: torch.Tensor, device: torch.device):
    B, T = waveforms.shape
    sr = SAMPLE_RATE
    durations = lengths.float() / sr
    log_durations = torch.log(durations + 1e-6)

    signs = torch.sign(waveforms)
    signs[signs == 0] = 1.0
    zcr = (torch.abs(torch.diff(signs, dim=-1)) > 0).float().sum(dim=-1) / (lengths.float() - 1 + 1e-8)

    n_fft, hop_length = 512, 256
    window = torch.hann_window(n_fft, device=device)
    S = torch.abs(torch.stft(waveforms, n_fft=n_fft, hop_length=hop_length, win_length=n_fft, window=window, return_complex=True)) + 1e-10
    power = S ** 2

    freqs = torch.linspace(0, sr / 2, n_fft // 2 + 1, device=device).view(1, -1, 1)
    spectral_centroid = ((freqs * S).sum(dim=1) / (S.sum(dim=1) + 1e-10)).mean(dim=-1)

    geom_mean = torch.exp(torch.mean(torch.log(power), dim=1))
    arith_mean = torch.mean(power, dim=1) + 1e-10
    spectral_flatness = (geom_mean / arith_mean).mean(dim=-1)

    freqs_1d = freqs.squeeze()
    band_mask = (freqs_1d >= 300) & (freqs_1d <= 3400)
    subband_energy_ratio = power[:, band_mask, :].sum(dim=(1, 2)) / (power.sum(dim=(1, 2)) + 1e-10)

    log_S = torch.log(S)
    cepstrum = torch.fft.irfft(log_S, n=n_fft, dim=1)
    q_min, q_max = max(1, int(sr / 400)), min(int(sr / 60), n_fft - 1)

    region = cepstrum[:, q_min:q_max, :]
    peak_vals, peak_rel_idx = torch.max(region, dim=1)
    peak_idx = peak_rel_idx + q_min

    fit_q = torch.arange(5, n_fft, device=device, dtype=torch.float32)
    X = torch.stack([fit_q, torch.ones_like(fit_q)], dim=-1)
    X_pinv = torch.linalg.pinv(X)
    fit_region = cepstrum[:, 5:, :]
    coeffs = torch.matmul(X_pinv.unsqueeze(0), fit_region)
    baseline_at_peak = coeffs[:, 0, :] * peak_idx.float() + coeffs[:, 1, :]
    cpp = (peak_vals - baseline_at_peak).mean(dim=-1)

    x = waveforms - waveforms.mean(dim=-1, keepdim=True)
    v = (torch.abs(x).mean(dim=-1) ** 2) / (x.pow(2).mean(dim=-1) + 1e-12)
    v_clipped = torch.clamp(v, float(_WADA_TABLE_V[0]), float(_WADA_TABLE_V[-1]))

    v_table_t = torch.from_numpy(_WADA_TABLE_V).float().to(device)
    snr_table_t = torch.from_numpy(_WADA_TABLE_SNR).float().to(device)

    indices = torch.bucketize(v_clipped, v_table_t) - 1
    indices = torch.clamp(indices, 0, len(_WADA_TABLE_V) - 2)
    v_0, v_1 = v_table_t[indices], v_table_t[indices + 1]
    s_0, s_1 = snr_table_t[indices], snr_table_t[indices + 1]
    weight = (v_clipped - v_0) / (v_1 - v_0 + 1e-12)
    wada_snr = s_0 + weight * (s_1 - s_0)

    qmat = torch.stack([
        log_durations, vads_mean, spectral_centroid, cpp,
        wada_snr, zcr, spectral_flatness, subband_energy_ratio
    ], dim=-1)

    return qmat.cpu().numpy()

In [ ]:
NOISE_TYPES = ["noise", "music", "babble"]
TEST_SNRS = [0, -5, -10, -15, -20]
TRAIN_SNR_RANGE = (-20, 0)

In [ ]:
def discover_voxceleb1_speakers_from_zip(zip_path: str, prefix: str = "wav/"):
    zf = _get_zip_handle(zip_path)
    speaker_to_files = {}

    for name in zf.namelist():
        if not name.endswith(".wav"):
            continue
        rel = name[len(prefix):] if name.startswith(prefix) else name
        parts = rel.split("/")
        if len(parts) < 2: continue
        speaker_id = parts[0]
        speaker_to_files.setdefault(speaker_id, []).append((zip_path, name))

    return speaker_to_files

def discover_generic_noise_files(root, subset):
    files = glob.glob(str(Path(root) / subset / "**" / "*.wav"), recursive=True)
    if not files:
        raise RuntimeError(f"No files found under {root}/{subset}")
    return files

def split_musan_disjoint(files, seed=42):
    rng = random.Random(seed)
    shuffled = list(files)
    rng.shuffle(shuffled)
    midpoint = len(shuffled)//2
    return shuffled[:midpoint], shuffled[midpoint:]

In [ ]:
def build_katav_benchmark_cache(args):
    print("Discovering VoxCeleb1 speakers via Drive fs...")
    train_speakers = discover_voxceleb1_speakers_from_zip(args["voxceleb1_train_zip"])
    test_speakers = discover_voxceleb1_speakers_from_zip(args["voxceleb1_test_zip"])
    print(f"  train: {len(train_speakers)} speakers, test: {len(test_speakers)} speakers")

    print("Discovering + splitting MUSAN (disjoint train/test halves)...")
    noise_pools_train, noise_pools_test = {}, {}
    musan_subset_names = {"noise":"noise", "music":"music", "babble":"speech"}
    for noise_type, subset in musan_subset_names.items():
        files = discover_generic_noise_files(args["musan_root"], subset)
        train_half, test_half = split_musan_disjoint(files, args["seed"])
        noise_pools_train[noise_type] = train_half
        noise_pools_test[noise_type] = test_half
        print(f"  {noise_type}: {len(train_half)} train, {len(test_half)} test")

    device_str = "cuda" if torch.cuda.is_available() else "cpu"
    device = torch.device(device_str)
    print(f"Using device: {device_str}")

    try:
        from speechbrain.inference.speaker import EncoderClassifier
    except ImportError:
        from speechbrain.pretrained import EncoderClassifier

    ecapa = EncoderClassifier.from_hparams(
        source=args["ecapa_model_path"], savedir="tmpdir_ecapa",
        run_opts={"device": device_str})
    ecapa = ecapa.to(device)

    cache_path = args["cache_path"]
    rows, noisy_embs, enhanced_embs, cos_dists, abs_diffs, quality_vecs = [], [], [], [], [], []
    clean_anchor_by_speaker = {}
    processed_keys = set()

    if os.path.exists(cache_path):
        print(f"Resuming from existing cache at {cache_path}")
        prev = torch.load(cache_path, weights_only=False)

        # ---- FIX: handle both DataFrame and list ----
        meta_data = prev.get("meta")
        if meta_data is None:
            meta_data = prev.get("meta_records")

        if meta_data is None:
            raise ValueError("No meta data found in cache file")

        if isinstance(meta_data, pd.DataFrame):
            rows = meta_data.to_dict("records")
        elif isinstance(meta_data, list):
            rows = meta_data
        else:
            raise TypeError(f"Unexpected type for meta: {type(meta_data)}")

        noisy_embs = list(prev.get("noisy_emb", []))
        enhanced_embs = list(prev.get("enhanced_emb", []))
        cos_dists = list(prev.get("cos_dist", []))
        abs_diffs = list(prev.get("abs_diff", []))
        quality_vecs = list(prev.get("quality_vec", []))
        clean_anchor_by_speaker = dict(prev.get("clean_anchor", {}))
        processed_keys = {(r["speaker"], r["file"], r["noise_type"], r["snr"]) for r in rows}
        print(f"Resuming with {len(rows)} rows.")

    def _file_ref_to_str(file_ref):
        return f"{file_ref[0]}!{file_ref[1]}"

    def _save_checkpoint():
        meta_ck = pd.DataFrame(rows)
        cache_ck = {
            "meta": meta_ck,
            "noisy_emb": np.stack(noisy_embs).astype(np.float32) if noisy_embs else np.zeros((0, EMBEDDING_DIM), np.float32),
            "enhanced_emb": np.stack(enhanced_embs).astype(np.float32) if enhanced_embs else np.zeros((0, EMBEDDING_DIM), np.float32),
            "cos_dist": np.array(cos_dists, dtype=np.float32),
            "abs_diff": np.stack(abs_diffs).astype(np.float32) if abs_diffs else np.zeros((0, EMBEDDING_DIM), np.float32),
            "quality_vec": np.stack(quality_vecs).astype(np.float32) if quality_vecs else np.zeros((0, len(FEATURE_ORDER)), np.float32),
            "clean_anchor": clean_anchor_by_speaker,
            "feature_names": FEATURE_ORDER,
        }
        torch.save(cache_ck, cache_path + ".tmp")
        shutil.move(cache_path + ".tmp", cache_path)
        if args.get("drive_cache_backup"):
            backup_dir = os.path.dirname(args["drive_cache_backup"])
            os.makedirs(backup_dir, exist_ok=True)
            shutil.copy2(cache_path, args["drive_cache_backup"])
        return meta_ck

    _t_start = time.time()
    _PROGRESS_EVERY = 200
    _CHECKPOINT_EVERY = args.get("cache_checkpoint_every_rows", 1000)
    _new_rows_since_start = 0

    BATCH_SIZE = args.get("batch_size", 32)
    compute_quality = args.get("compute_quality", True)

    # -------------------------------------------------------------
    # process_speaker_set (unchanged)
    # -------------------------------------------------------------
    def process_speaker_set(speakers, split_name, noise_pools, snr_mode):
        nonlocal _new_rows_since_start

        for spk, files in speakers.items():
            utts = sorted(files, key=_file_ref_to_str)[:args["utts_per_speaker"]]
            if len(utts) < 2:
                continue
            key = ("VoxCeleb1", spk)
            if key not in clean_anchor_by_speaker:
                clean = load_mono_16k_from_zip(utts[0])
                emb = get_embedding(clean, ecapa, device)
                clean_anchor_by_speaker[key] = emb

        work_items = []
        for spk, files in speakers.items():
            utts = sorted(files, key=_file_ref_to_str)[:args["utts_per_speaker"]]
            if len(utts) < 2:
                continue
            for utt in utts:
                utt_str = _file_ref_to_str(utt)
                file_cond_rng = per_file_rng("train_conditions", utt_str, args["seed"])
                conditions_this_utt = (
                    [(nt, snr) for nt in NOISE_TYPES for snr in TEST_SNRS]
                    if snr_mode == "grid" else
                    [(file_cond_rng.choice(NOISE_TYPES), round(file_cond_rng.uniform(*TRAIN_SNR_RANGE), 2))
                    for _ in range(args["train_augmentations_per_utt"])]
                )
                for noise_type, snr in conditions_this_utt:
                    if (spk, utt_str, noise_type, snr) not in processed_keys:
                        work_items.append((spk, utt, noise_type, snr))

        print(f"Processing {len(work_items)} total tasks for {split_name} split...")

        def prepare_single_audio(item_tuple):
            spk, utt, noise_type, snr = item_tuple
            utt_str = _file_ref_to_str(utt)
            rng = per_file_rng(f"{noise_type}_{snr}", utt_str, args["seed"])
            noise_path = rng.choice(noise_pools[noise_type])
            clean = load_mono_16k_from_zip(utt)
            degraded = add_noise_at_snr(clean, noise_path, snr, rng)
            enhanced, vad_flags = enhance_waveform_cpu(degraded, return_vad=True)
            vad_mean = float(np.mean(vad_flags)) if len(vad_flags) else 0.0
            return (degraded, enhanced, vad_mean, noise_type, snr, noise_path, utt_str, spk)

        with ThreadPoolExecutor(max_workers=4) as executor:
            for start_idx in range(0, len(work_items), BATCH_SIZE):
                chunk = work_items[start_idx:start_idx + BATCH_SIZE]
                batch_items = list(executor.map(prepare_single_audio, chunk))
                if not batch_items:
                    continue

                degraded_list = [item[0] for item in batch_items]
                enhanced_list = [item[1] for item in batch_items]
                vads = [item[2] for item in batch_items]
                meta_info = [(item[3], item[4], item[5], item[6], item[7]) for item in batch_items]

                deg_lengths = torch.tensor([w.shape[-1] for w in degraded_list], device=device)
                enh_lengths = torch.tensor([w.shape[-1] for w in enhanced_list], device=device)
                max_len_deg = min(max(deg_lengths).item(), MAX_SAMPLES)
                max_len_enh = min(max(enh_lengths).item(), MAX_SAMPLES)

                padded_degraded = torch.stack([
                    F.pad(w[:MAX_SAMPLES], (0, max_len_deg - min(w.shape[-1], MAX_SAMPLES))) for w in degraded_list
                ], dim=0).float().to(device)

                padded_enhanced = torch.stack([
                    F.pad(w[:MAX_SAMPLES], (0, max_len_enh - min(w.shape[-1], MAX_SAMPLES))) for w in enhanced_list
                ], dim=0).float().to(device)

                with torch.no_grad():
                    noisy_embs_batch = ecapa.encode_batch(padded_degraded)
                    enhanced_embs_batch = ecapa.encode_batch(padded_enhanced)
                    if compute_quality:
                        vad_t = torch.tensor(vads, device=device)
                        qvecs_batch = compute_spectral_quality_features_batch(
                            padded_degraded, deg_lengths, vad_t, device
                        )
                    else:
                        qvecs_batch = np.zeros((len(batch_items), len(FEATURE_ORDER)), dtype=np.float32)

                for i, (noise_type, snr, noise_path, utt_str, spk) in enumerate(meta_info):
                    n_emb = noisy_embs_batch[i].cpu().flatten()
                    e_emb = enhanced_embs_batch[i].cpu().flatten()
                    cos_sim = torch.dot(n_emb, e_emb) / (torch.norm(n_emb) * torch.norm(e_emb) + 1e-8)
                    cos_dist = float((1 - cos_sim).item())
                    abs_diff = (n_emb - e_emb).abs().numpy()

                    rows.append({
                        "language": "VoxCeleb1", "speaker": spk, "file": utt_str,
                        "condition": f"{noise_type}_{snr}db", "noise_type": noise_type,
                        "snr": snr, "split": split_name, "noise_path": noise_path,
                    })

                    noisy_embs.append(n_emb.numpy())
                    enhanced_embs.append(e_emb.numpy())
                    cos_dists.append(cos_dist)
                    abs_diffs.append(abs_diff)
                    quality_vecs.append(qvecs_batch[i])
                    processed_keys.add((spk, utt_str, noise_type, snr))
                    _new_rows_since_start += 1

                del padded_degraded, padded_enhanced
                if compute_quality:
                    del qvecs_batch
                torch.cuda.empty_cache()

                if len(rows) % _PROGRESS_EVERY < BATCH_SIZE:
                    elapsed = time.time() - _t_start
                    rate = _new_rows_since_start / elapsed if elapsed > 0 else 0
                    print(f"   ... {len(rows)} processed | elapsed={elapsed/60:.1f}min | speed={rate:.2f} files/sec")

                if _new_rows_since_start % _CHECKPOINT_EVERY < BATCH_SIZE:
                    _save_checkpoint()
                    print(f"   [checkpoint saved: {len(rows)} rows]")

    # ------------------------------------------------------------------
    # Split speakers and run
    # ------------------------------------------------------------------
    train_spk_items = list(train_speakers.items())
    random.Random(args["seed"]).shuffle(train_spk_items)
    n_val = max(1, int(0.1 * len(train_spk_items)))
    val_speakers = dict(train_spk_items[:n_val])
    actual_train_speakers = dict(train_spk_items[n_val:])

    print("\nProcessing train speakers...")
    process_speaker_set(actual_train_speakers, "train", noise_pools_train, "random")
    process_speaker_set(val_speakers, "val", noise_pools_train, "random")
    print("\nProcessing test speakers...")
    process_speaker_set(test_speakers, "test", noise_pools_test, "grid")

    _save_checkpoint()
    return torch.load(cache_path, weights_only=False)

In [ ]:
!cp /content/drive/MyDrive/Data_Science_Project/full_cache/katav_benchmark_cache_fixed.pt /content/

In [ ]:
MUSAN_ROOT = "/content/musan"
ECAPA_PATH = "/content/drive/MyDrive/Data_Science_Project/Models/ECAPA-TDNN/"

args = {
    "voxceleb1_train_zip": '/content/drive/MyDrive/Data_Science_Project/data/vox1_dev_wav.zip',
    "voxceleb1_test_zip": '/content/drive/MyDrive/Data_Science_Project/data/vox1_test_wav.zip',
    "musan_root": MUSAN_ROOT,
    "ecapa_model_path": ECAPA_PATH,
    "utts_per_speaker": 999,
    "train_augmentations_per_utt": 3,
    "seed": 42,
    "cache_path": "/content/katav_benchmark_cache_fixed.pt",
    "cache_checkpoint_every_rows": 1000,
    "drive_cache_backup": "/content/cache/katav_benchmark_cache_full.pt",
    "batch_size": 128,               # lowered for stability
    "compute_quality": True,        # set to False if you need speed
}

cache = build_katav_benchmark_cache(args)

In [ ]:
import torch

cache = torch.load("/content/katav_benchmark_cache_fixed.pt", weights_only=False)

In [ ]:
cache.items()

Buffered data was truncated after reaching the output size limit.